# Hunting statistics → per-year files for ArcGIS join

Prepares the Czech hunting-statistics workbook for joining to the per-year hunting-ground
shapefiles in ArcGIS, with **English column names**, and verifies the result.

**What it does**
1. Loads the source (handles the 3-row banner header automatically).
2. Keeps only `ROK`, `honitba`, and the three requested metric families:
   `PLANLOVU_` (hunting plan), `ODSTREL_` (hunting bag / actually shot), `JKS_` (spring counts).
3. Renames the Czech field codes to English `Metric_Species_Class` (e.g. `Plan_RedDeer_Male`),
   and writes a data dictionary mapping English ↔ Czech.
4. Converts the literal text `"NULL"` to empty (true missing) while keeping real `0` values.
5. Writes one CSV per year (`harvest_<year>.csv`) — a clean 1:1 join target for that year's shapefile.
6. Runs verification checks so you can confirm nothing was scrambled.

`honitba` is **kept unchanged** — it is the join key and must match the shapefile field `HONITBA`.
`ROK` is renamed to `year`.

In [1]:
# ===== CONFIG =====
SOURCE   = r"Hunting statistics_2003_2022.xlsx"   # .xlsx (authoritative) or a .csv export
OUTDIR   = r"per_year"                            # folder for the per-year CSVs
YEAR_COL = "ROK"
ID_COL   = "honitba"
METRIC_PREFIXES = ("PLANLOVU_", "ODSTREL_", "JKS_")   # plan, bag(actual), spring counts
HEADER_ROW_XLSX = 4   # 1-based row with the machine codes (ROK, honitba, PLANLOVU_JELEN, ...)

# Short English labels for each metric family (edit to taste):
METRIC_LABELS = {"PLANLOVU_": "Plan", "ODSTREL_": "Bag", "JKS_": "Spring"}
# NOTE: trailing predator/invasive columns (MYVAL_LOV, NOREK_LOV, ...) have no PLANLOVU_/ODSTREL_/JKS_
# prefix and are excluded automatically. Add a prefix above if you ever want them.

In [2]:
import os, re, unicodedata, pandas as pd, numpy as np
from openpyxl import load_workbook
os.makedirs(OUTDIR, exist_ok=True)

def load_selected(path, prefixes, year_col, id_col, header_row_xlsx=4):
    """Stream the source; return (df_selected, banner). banner maps each kept metric column
    -> (species, class) when the 3-row banner is present, else None (e.g. a single-header CSV)."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xlsm"):
        wb = load_workbook(path, read_only=True); ws = wb.active
        it = ws.iter_rows(values_only=True)
        top = [next(it) for _ in range(header_row_xlsx)]
        header  = list(top[-1])                                # machine-code row
        species = list(top[1]) if header_row_xlsx >= 2 else None
        klass   = list(top[2]) if header_row_xlsx >= 3 else None
        ci = {n: j for j, n in enumerate(header)}
        keep = [ci[year_col], ci[id_col]] + [j for j, n in enumerate(header)
                if isinstance(n, str) and n.startswith(prefixes)]
        names = [header[j] for j in keep]
        rows = []
        for r in it:
            if r[ci[year_col]] is None and r[ci[id_col]] is None: continue
            rows.append([r[j] for j in keep])
        wb.close()
        df = pd.DataFrame(rows, columns=names)
        banner = {header[j]: (species[j] if species else None, klass[j] if klass else None)
                  for j in keep if j not in (ci[year_col], ci[id_col])}
        return df, banner
    # ---- CSV branch ----
    for hdr in (0, 3):
        df = pd.read_csv(path, header=hdr, dtype={id_col: str}, low_memory=False)
        if year_col in df.columns and id_col in df.columns:
            keep = [year_col, id_col] + [c for c in df.columns if str(c).startswith(prefixes)]
            return df[keep].copy(), None
    raise ValueError("Could not find ROK/honitba header in CSV")

In [3]:
df, banner = load_selected(SOURCE, METRIC_PREFIXES, YEAR_COL, ID_COL, HEADER_ROW_XLSX)
metric_cols = [c for c in df.columns if c not in (YEAR_COL, ID_COL)]
print("Loaded shape:", df.shape)
print("Metric columns kept:", len(metric_cols))
print("Banner (species/class map) available:", banner is not None)

Loaded shape: (115057, 162)
Metric columns kept: 160
Banner (species/class map) available: True


## Build English column names

`Metric_Species_Class`, ASCII and ArcGIS-safe (no spaces or diacritics). A dictionary file
`_column_dictionary.csv` records the English ↔ Czech mapping for the paper's methods.

In [4]:
def slug(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = re.sub(r"[^0-9A-Za-z]+", " ", s).strip()
    return "".join(w[:1].upper() + w[1:] for w in s.split())   # CamelCase

rename_map = {}
if banner is None:
    print("No banner header -> keeping original Czech codes (English names need the 3-row header).")
    col_dict = None
else:
    seen, recs = {}, []
    for code in metric_cols:
        sp, kl = banner[code]
        pref   = next(p for p in METRIC_PREFIXES if code.startswith(p))
        metric = METRIC_LABELS.get(pref, pref.strip("_"))
        name   = f"{metric}_{slug(sp)}_{slug(kl)}"
        if name in seen:                       # collision guard -> append czech code
            name = f"{name}_{code}"
        seen[name] = code
        rename_map[code] = name
        recs.append({"english": name, "czech_code": code,
                     "metric": metric, "species": str(sp).strip(), "class": str(kl).strip()})
    col_dict = pd.DataFrame(recs)
    col_dict.to_csv(os.path.join(OUTDIR, "_column_dictionary.csv"), index=False)

    df = df.rename(columns=rename_map)
    metric_cols = [rename_map[c] for c in metric_cols]
    df = df.rename(columns={YEAR_COL: "year"}); YEAR_COL = "year"   # honitba kept as-is (join key)
    print(f"Renamed {len(rename_map)} columns. Dictionary -> {OUTDIR}/_column_dictionary.csv")
    print("Examples:", *metric_cols[:6], sep="\n  ")
col_dict.head() if banner is not None else None

Renamed 160 columns. Dictionary -> per_year/_column_dictionary.csv
Examples:
  Plan_RedDeer_Male
  Plan_RedDeer_Female
  Plan_RedDeer_Juvenile
  Plan_RedDeer_Total
  Plan_FallowDeer_Male
  Plan_FallowDeer_Female


,english,czech_code,metric,species,class
0,Plan_RedDeer_Male,PLANLOVU_JELEN,Plan,Red deer,Male
1,Plan_RedDeer_Female,PLANLOVU_LAN,Plan,Red deer,Female
2,Plan_RedDeer_Juvenile,PLANLOVU_KOLOUCH,Plan,Red deer,Juvenile
3,Plan_RedDeer_Total,PLANLOVU_JELENISA,Plan,Red deer,Total
4,Plan_FallowDeer_Male,PLANLOVU_DANEK,Plan,Fallow deer,Male


## Clean: `NULL` text → missing, keep real zeros, fix types

In [5]:
df[ID_COL] = df[ID_COL].astype(str).str.strip()

na_before = df[metric_cols].isna().sum().sum()
df[metric_cols] = df[metric_cols].replace("NULL", np.nan)
na_after_null = df[metric_cols].isna().sum().sum()
df[metric_cols] = df[metric_cols].apply(pd.to_numeric, errors="coerce")
na_after_num = df[metric_cols].isna().sum().sum()

df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64")

print(f"Missing cells  | before: {na_before:>9}  | after NULL->NaN: {na_after_null:>9}  | after numeric: {na_after_num:>9}")
print(f"Unexpected coercions (non-NULL strings turned to missing): {na_after_num - na_after_null}")
assert na_after_num - na_after_null == 0, "Some non-NULL values failed numeric conversion - inspect!"
print("Real zeros preserved as 0; only literal 'NULL' became empty.")

C:\Users\Prosper\AppData\Local\Temp\ipykernel_44380\925584237.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[metric_cols] = df[metric_cols].replace("NULL", np.nan)


Missing cells  | before:         0  | after NULL->NaN:   3943448  | after numeric:   3943448
Unexpected coercions (non-NULL strings turned to missing): 0
Real zeros preserved as 0; only literal 'NULL' became empty.


## Write one CSV per year
Each file is a clean 1:1 join target. Missing values are written empty (ArcGIS reads them as null, not 0).

In [6]:
written = {}
for yr, g in df.groupby(YEAR_COL):
    fp = os.path.join(OUTDIR, f"harvest_{int(yr)}.csv")
    g.to_csv(fp, index=False, na_rep="")
    written[int(yr)] = (fp, len(g))
print(f"Wrote {len(written)} files to {OUTDIR}/")
for yr in sorted(written): print(f"  harvest_{yr}.csv  ({written[yr][1]} rows)")

Wrote 20 files to per_year/
  harvest_2003.csv  (5618 rows)
  harvest_2004.csv  (5682 rows)
  harvest_2005.csv  (5712 rows)
  harvest_2006.csv  (5725 rows)
  harvest_2007.csv  (5710 rows)
  harvest_2008.csv  (5728 rows)
  harvest_2009.csv  (5740 rows)
  harvest_2010.csv  (5733 rows)
  harvest_2011.csv  (5744 rows)
  harvest_2012.csv  (5751 rows)
  harvest_2013.csv  (5789 rows)
  harvest_2014.csv  (5792 rows)
  harvest_2015.csv  (5804 rows)
  harvest_2016.csv  (5815 rows)
  harvest_2017.csv  (5793 rows)
  harvest_2018.csv  (5784 rows)
  harvest_2019.csv  (5782 rows)
  harvest_2020.csv  (5786 rows)
  harvest_2021.csv  (5787 rows)
  harvest_2022.csv  (5782 rows)


# Verification
Anything other than the expected `PASS` line means stop and inspect.

In [7]:
# V1 - round-trip integrity: re-read each file and compare to the in-memory cleaned data
fails = []
for yr, (fp, n) in written.items():
    rt  = pd.read_csv(fp, dtype={ID_COL: str})
    src = df[df[YEAR_COL] == yr].reset_index(drop=True)
    ok = (len(rt) == len(src)
          and (rt[ID_COL].values == src[ID_COL].values).all()
          and np.array_equal(np.nan_to_num(rt[metric_cols].to_numpy("float64"), nan=-9e9),
                             np.nan_to_num(src[metric_cols].to_numpy("float64"), nan=-9e9)))
    if not ok: fails.append(yr)
print("V1 round-trip:", "PASS - all files match source exactly" if not fails else f"FAIL in years {fails}")

V1 round-trip: PASS - all files match source exactly


In [8]:
# V2 - honitba unique and non-empty within each year (required for a 1:1 join)
dup = [yr for yr,(fp,n) in written.items()
       if pd.read_csv(fp, dtype={ID_COL:str})[ID_COL].duplicated().any()]
nul = [yr for yr,(fp,n) in written.items()
       if pd.read_csv(fp, dtype={ID_COL:str})[ID_COL].isin(["", "nan", "None"]).any()]
print("V2 uniqueness:", "PASS - no duplicate IDs" if not dup else f"FAIL duplicates in {dup}")
print("V2 non-empty :", "PASS - no empty IDs"     if not nul else f"FAIL empty IDs in {nul}")

V2 uniqueness: PASS - no duplicate IDs
V2 non-empty : PASS - no empty IDs


In [9]:
# V3 - row reconciliation + coverage table (feeds the Fig.2-style coverage matrix later)
total_rows = sum(n for _, n in written.values())
print("V3 reconciliation:", "PASS" if total_rows == len(df) else "FAIL",
      f"(sum per-year={total_rows}, cleaned df={len(df)})")
coverage = pd.DataFrame(
    {yr: [n, pd.read_csv(fp, dtype={ID_COL:str})[ID_COL].nunique()]
     for yr,(fp,n) in sorted(written.items())},
    index=["rows", "distinct_ids"]).T
coverage.index.name = "year"
coverage.to_csv(os.path.join(OUTDIR, "_coverage_by_year.csv"))
coverage

V3 reconciliation: PASS (sum per-year=115057, cleaned df=115057)


,rows,distinct_ids
year,,
2003,5618,5618
2004,5682,5682
2005,5712,5712
2006,5725,5725
2007,5710,5710
2008,5728,5728
2009,5740,5740
2010,5733,5733
2011,5744,5744


In [10]:
# V4 - every file shares the identical column schema
schemas = {tuple(pd.read_csv(fp, nrows=0).columns) for fp,_ in written.values()}
print("V4 schema consistency:", "PASS - identical columns in all files" if len(schemas)==1 else "FAIL - schemas differ")
print("Columns per file:", len(next(iter(schemas))))

V4 schema consistency: PASS - identical columns in all files
Columns per file: 162


In [11]:
# V5 - value composition per year (sanity: the NULL/zero mix should be stable, not jump around)
comp = []
for yr,(fp,n) in sorted(written.items()):
    m = pd.read_csv(fp, dtype={ID_COL:str})[metric_cols]; cells = m.size
    comp.append([yr, round(m.isna().sum().sum()/cells*100,1),
                 round((m.to_numpy()==0).sum()/cells*100,1),
                 round((m.to_numpy()>0).sum()/cells*100,1)])
pd.DataFrame(comp, columns=["year","pct_empty_NULL","pct_zero","pct_positive"])

,year,pct_empty_NULL,pct_zero,pct_positive
0,2003,13.1,68.5,18.4
1,2004,13.1,68.2,18.7
2,2005,13.1,68.2,18.7
3,2006,13.1,68.4,18.5
4,2007,13.1,67.7,19.2
5,2008,13.1,67.6,19.2
6,2009,13.1,67.7,19.1
7,2010,68.1,12.7,19.2
8,2011,68.5,12.6,18.9
9,2012,68.5,12.5,19.0


In [12]:
# V6 - source consistency: do sex/age components sum to the species Total?
# Uses the banner; maps Czech codes through rename_map to current English columns. Skipped if no banner.
if banner is None:
    print("V6 skipped - no banner header available.")
else:
    comp_classes = {"Male","Female","Juvenile","Subadult","Yearling"}
    cur = lambda code: rename_map.get(code, code)     # czech code -> current df column
    groups = {}
    for code,(sp,kl) in banner.items():
        pref = next((p for p in METRIC_PREFIXES if code.startswith(p)), None)
        if pref is None or sp is None or kl is None: continue
        groups.setdefault((pref, str(sp).strip()), {})[str(kl).strip()] = code
    viol = []
    for (pref, sp), cls in groups.items():
        if "Total" not in cls: continue
        parts = [cur(c) for k,c in cls.items() if k in comp_classes]
        if not parts: continue
        sub = df[parts + [cur(cls["Total"])]].dropna()
        if sub.empty: continue
        bad = int((sub[parts].sum(axis=1) != sub[cur(cls["Total"])]).sum())
        if bad: viol.append((METRIC_LABELS.get(pref,pref), sp, bad, len(sub)))
    if not viol:
        print("V6 component-vs-Total: PASS - components sum to Total wherever both are reported.")
    else:
        print("V6 component-vs-Total: rows where components != Total (may reflect the source data, not your prep):")
        for m, sp, bad, tot in sorted(viol, key=lambda x:-x[2]):
            print(f"  {m:7s} {sp:22s} {bad:>6} / {tot} reported rows")

V6 component-vs-Total: rows where components != Total (may reflect the source data, not your prep):
  Plan    Sika deer Japaneese      3807 / 98472 reported rows


# Joining in ArcGIS Pro

For each year **2017–2022** (table + polygons both exist):

1. Add the year's shapefile and `harvest_<year>.csv`.
2. Confirm both keys are **Text**: `HONITBA` (shapefile) and `honitba` (CSV).
3. Shapefile → **Joins and Relates → Add Join**, target `HONITBA`, join field `honitba`,
   tick **Validate Join** (its match rate is your per-year coverage number).
4. **Export Features → a File Geodatabase feature class** (not a shapefile — shapefiles truncate
   field names to 10 characters, which would mangle names like `Spring_WhiteTailedDeer_Female`).

**Pre-2017 years (no shapefile):** join those CSVs to the **2017** polygons by `honitba`; only IDs
that still existed in 2017 will match. `_coverage_by_year.csv` plus the validated match rate tell
you how much of each old year you can place.

**Outputs in `per_year/`:** the 20 `harvest_<year>.csv` files, `_column_dictionary.csv`
(English ↔ Czech field map), and `_coverage_by_year.csv`.

**Two flags:** `nehonební` polygons (`A`-coded) find no match and drop out - fine. `obora`
(fenced enclosure) rows carry real codes and will join, so add a category field if you want to
separate enclosures from free-ranging game later.